In [38]:
try :
    from elasticsearch import Elasticsearch
    import os
    from IPython.display import display
    import sys
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    import seaborn as sns
    from datetime import datetime
    from sklearn.naive_bayes import GaussianNB
    from sklearn.metrics import confusion_matrix
    import matplotlib.pyplot as plt
    import seaborn as sn
    import numpy as np
    from sklearn.tree import DecisionTreeClassifier
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.svm import SVC
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.metrics import classification_report
    from sklearn.model_selection import GridSearchCV
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.metrics import accuracy_score,f1_score
    from sklearn.metrics import classification_report
    from sklearn.metrics import confusion_matrix
    from sklearn.feature_selection import SelectFromModel
    from sklearn.model_selection import train_test_split
    from catboost import Pool
    from catboost import CatBoostClassifier
except Exception as e :
    print("Some Modules are Missing{}".format(e))

In [39]:
## with out srcip and url

In [40]:
all_features = ['_source.data.protocol',
                         '_source.data.id',
                         '_source.rule.firedtimes',
                         '_source.rule.mail',
                         '_source.rule.level',
                         '_source.rule.description',
                         '_source.rule.groups',
                         '_source.rule.pci_dss',
                         '_source.rule.tsc',
                         '_source.rule.nist_800_53',
                         '_source.rule.gdpr',
                         '_source.rule.mitre.id',
                         '_source.rule.frequency',
                         '_source.rule.hipaa',
                         '_source.agent.description',
                        '_source.rule.id',
                        'output_1',
                        'output_2' ]
category_features= ['_source.data.protocol',
                         '_source.data.id',
                         '_source.rule.mail',
                         '_source.rule.description',
                         '_source.rule.groups',
                         '_source.rule.pci_dss',
                         '_source.rule.tsc',
                         '_source.rule.nist_800_53',
                         '_source.rule.gdpr',
                         '_source.rule.mitre.id',
                         '_source.rule.hipaa',
                         '_source.agent.description',
                        '_source.rule.id',

                         '_source.rule.frequency']

numerical_features = ['_source.rule.firedtimes',
                      '_source.rule.level'
                     ]


param_grid_1 = {'learning_rate': [0.03,0.07,0.1],
                'depth': [6,8],
                'l2_leaf_reg': [7,9]
       }


param_grid_2= {'learning_rate': [0.03,0.07,0.1],
                'depth': [6,8],
               'l2_leaf_reg': [7,9]
       }

In [41]:
def preprocessing(df):  
    #these category features if they didn't exist we will add them to the dataframe
    list_test_columns= list(df.columns)
    for i in range ( 0, len(category_features)):
        if(category_features[i] not in list_test_columns):
            df[category_features[i]] = " "
    for c in numerical_features:
        if(c not in list_test_columns):
            df[c] = 1
    # fill the columns that has None values with empty strings.
    for c in category_features:
        if(df[c].isnull().any()):
            df[c] = df[c].fillna(' ')
    # fill the columns rule level with rule.level = 3
    df["_source.rule.level"] = df["_source.rule.level"].fillna(3)
    df["_source.rule.firedtimes"] = df["_source.rule.firedtimes"].fillna(1)
    
    
    # fill mail values with 1 and 0 
    for index,row in df.iterrows():
        if(isinstance(row["_source.rule.id"],int)):
            df.at[index,'_source.rule.id'] = str(row['_source.rule.id'])

        
        for cat_fe in category_features:
            if(isinstance(row[cat_fe],list)):
                df.at[index,cat_fe] = str(row[cat_fe])
        if (row["_source.rule.mail"]==True):
            df.at[index,"_source.rule.mail"] = 1
        else :
            df.at[index,"_source.rule.mail"] = 0
    df["_source.agent.description"] = "APACHE_SERVER"
    df = df.fillna(' ')
    return df

In [42]:
## read the data
def train_multi_classifier_model(path_train,path_test, save_model= False):
    dataset_training = pd.read_csv(path_train)
    dataset_testing = pd.read_csv(path_test)
    dataset_training = preprocessing(dataset_training)
    dataset_testing = preprocessing(dataset_testing)

    dataset_training = dataset_training[all_features]
    dataset_testing = dataset_testing[all_features]

    y1_train = dataset_training["output_1"]
    X1_train = dataset_training.drop(['output_1','output_2'], axis=1)
    train_pool_1 =Pool(
        data=X1_train,
        label=y1_train,
        cat_features = category_features,
    )
    
    
    y1_validation = dataset_testing["output_1"]
    X1_validation = dataset_testing.drop(['output_1','output_2'], axis=1)
      
    test_pool_1 = Pool(
        data=X1_validation,
        label=y1_validation,
        cat_features=category_features,
    )
    model_1 = CatBoostClassifier(
       depth= 8,
    l2_leaf_reg= 7, 
    iterations= 2000,
    learning_rate= 0.05,
        task_type="GPU"
    )
    model_1.fit(
        train_pool_1,
        eval_set=test_pool_1,
        verbose=False,
        plot=True
    );
    dataset_training_attack = dataset_training[dataset_training["output_2"] != "NORMAL" ]
    y2_train= dataset_training_attack["output_2"]
    X2_train = dataset_training_attack.drop(['output_1','output_2'], axis=1)  
    train_pool_2 =Pool(
        data=X2_train,
        label=y2_train,
        cat_features = category_features,
    )

    dataset_testing_attack = dataset_testing[dataset_testing["output_2"] != "NORMAL" ]
    y2_validation= dataset_testing_attack["output_2"]
    X2_validation = dataset_testing_attack.drop(['output_1','output_2'], axis=1)  
    test_pool_2 = Pool(
        data=X2_validation,
        label=y2_validation,
        cat_features=category_features,
    )
    model_2 = CatBoostClassifier(
        depth= 6,
    l2_leaf_reg= 7, 
    learning_rate= 0.05,
    iterations=5000,
        task_type="GPU"
    )
    model_2.fit(
        train_pool_2,
        eval_set=test_pool_2,
        verbose=False,
        plot=True
    );
    y1_predict = model_1.predict(X1_validation)
    y2_predict = model_2.predict(X2_validation)
    
    res = []
    for item in y2_predict:
        res.append(item[0])
    y2_predict=np.array(res)
    
    print("*"*40,"the first model report","*"*40)
    display(pd.crosstab(y1_validation,y1_predict))
    print("*"*60)
    print(classification_report(y1_validation, y1_predict))

    
    
    print("*"*40,"the second model report","*"*40)
    display(pd.crosstab(y2_validation,y2_predict))
    print("*"*60)
    print(classification_report(y2_validation, y2_predict))
    
    if(save_model):
        model_1.save_model("../models/catboost_model_1.bin")
        model_1.save_model("../models/catboost_model_1.json",format='json')
        model_2.save_model("../models/catboost_model_2.bin")
        model_2.save_model("../models/catboost_model_2.json",format='json')
    



    


In [43]:
def train_first_classifier_model_grid_search(path_train,path_test, save_model=False):
    # read training set and testing set.
    dataset_training = pd.read_csv(path_train)
    dataset_testing = pd.read_csv(path_test)
    dataset_training = preprocessing(dataset_training)
    dataset_testing = preprocessing(dataset_testing)
    
    # slice the training dataset and testing dataset to take the columns
    dataset_training = dataset_training[all_features]
    dataset_testing = dataset_testing[all_features]
    
    # take the features and leave the class
    y1_train = dataset_training["output_1"]
    X1_train = dataset_training.drop(['output_1','output_2'], axis=1)
    train_pool_1 =Pool(
        data=X1_train,
        label=y1_train,
        cat_features = category_features,
    )
    
    
    y1_validation = dataset_testing["output_1"]
    X1_validation = dataset_testing.drop(['output_1','output_2'], axis=1)
      
    test_pool_1 = Pool(
        data=X1_validation,
        label=y1_validation,
        cat_features=category_features,
    )
    
    # hyper parameter tunning for CatBoost Classifier
    model_1 = CatBoostClassifier(logging_level='Silent' , task_type="GPU")
    grid_search_model_1_results= model_1.grid_search(param_grid_1,
                                                    train_pool_1,
                                                    cv=3,
                                                    partition_random_seed=0,
                                                    calc_cv_statistics=True,
                                                    search_by_train_test_split=True,
                                                    refit=True,
                                                    shuffle=True,
                                                    stratified=True,
                                                    train_size=0.8,
                                                    plot=True,
                                                    verbose=False
                                                    )
    
    
    
    
    y1_predict = model_1.predict(X1_validation)

    ## grid search results for model_1
    print("*"*40,"the first model Gridsearch results","*"*40)
    print("*"*20,"cross validation results ","*"*20)
    display(pd.DataFrame(grid_search_model_1_results["cv_results"]))
    print("*"*20,"best parameters","*"*20)
    display(grid_search_model_1_results["params"])
    
    
    print("*"*40,"the first model report","*"*40)
    display(pd.DataFrame(pd.crosstab(y1_validation,y1_predict)))
    print("*"*60)
    print(classification_report(y1_validation, y1_predict))
    
    if(save_model):
        model_1.save_model("../models/catboost_model_gridsearch_1.bin")
        model_1.save_model("../models/catboost_model_gridsearch_1.json",format='json')


In [44]:
def train_second_classifier_model_grid_search(path_train,path_test, save_model=False):
    # read training set and testing set.
    dataset_training = pd.read_csv(path_train)
    dataset_testing = pd.read_csv(path_test)
    dataset_training = preprocessing(dataset_training)
    dataset_testing = preprocessing(dataset_testing)
    
    # slice the training dataset and testing dataset to take the columns
    dataset_training = dataset_training[all_features]
    dataset_testing = dataset_testing[all_features]
    
    # take the features and leave the class
    dataset_training_attack = dataset_training[dataset_training["output_2"] != "NORMAL" ]
    y2_train= dataset_training_attack["output_2"]
    X2_train = dataset_training_attack.drop(['output_1','output_2'], axis=1)  
    train_pool_2 =Pool(
        data=X2_train,
        label=y2_train,
        cat_features = category_features,
    )

    dataset_testing_attack = dataset_testing[dataset_testing["output_2"] != "NORMAL" ]
    y2_validation= dataset_testing_attack["output_2"]
    X2_validation = dataset_testing_attack.drop(['output_1','output_2'], axis=1)  
    test_pool_2 = Pool(
        data=X2_validation,
        label=y2_validation,
        cat_features=category_features,
    )
    
    # Hyper Parameter tunning for CatBoost Model 
    model_2 = CatBoostClassifier(logging_level='Silent',task_type="GPU")
    grid_search_model_2_results=model_2.grid_search(param_grid_2,
                                                    train_pool_2,
                                                    cv=3,
                                                    partition_random_seed=0,
                                                    calc_cv_statistics=True,
                                                    search_by_train_test_split=True,
                                                    refit=True,
                                                    shuffle=True,
                                                    stratified=True,
                                                    train_size=0.8,
                                                     plot=True,
                                                    verbose=False
                                                   )                                          
    
    
    
    # predict 
    y2_predict = model_2.predict(X2_validation)
    
    res = []
    for item in y2_predict:
        res.append(item[0])
    y2_predict=np.array(res)
    

        ## grid search results for model_2
    print("*"*40,"the second model Gridsearch results","*"*40)
    print("*"*20,"cross validation results ","*"*20)
    display(pd.DataFrame(grid_search_model_2_results["cv_results"]))
    print("*"*20,"best parameters","*"*20)
    display(grid_search_model_2_results["params"])
    
    print("*"*40,"the second model report","*"*40)
    display(pd.crosstab(y2_validation,y2_predict))
    print("*"*60)
    print(classification_report(y2_validation, y2_predict))
      
    if(save_model):
        model_2.save_model("../models/catboost_model_gridsearch_2.bin")
        model_2.save_model("../models/catboost_model_gridsearch_2.json",format='json')
    

In [34]:
path_training = "../datasets/training/final_training_dataset_no_history_SMOTENC.csv"
path_testing = "../datasets/testing/final_testing_dataset_no_history.csv"
train_first_classifier_model_grid_search(path_train=path_training,path_test=path_testing,save_model=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

**************************************** the first model Gridsearch results ****************************************
******************** cross validation results  ********************


,iterations,test-Logloss-mean,test-Logloss-std,train-Logloss-mean,train-Logloss-std
0,0,0.422337,0.008267,0.419495,0.006920
1,1,0.384902,0.006111,0.380092,0.004085
2,2,0.368628,0.007980,0.361492,0.004353
3,3,0.358041,0.002337,0.350438,0.002648
4,4,0.356800,0.001658,0.347639,0.004103
...,...,...,...,...,...
995,995,0.349337,0.015639,0.225168,0.007876
996,996,0.349486,0.015809,0.225120,0.007852
997,997,0.349311,0.016158,0.225066,0.007861
998,998,0.349265,0.016161,0.225025,0.007888


******************** best parameters ********************


{'depth': 8, 'l2_leaf_reg': 9, 'learning_rate': 0.7}

**************************************** the first model report ****************************************


col_0,ATTACK,NORMAL
output_1,,
ATTACK,6478,1543
NORMAL,229,1041


************************************************************
              precision    recall  f1-score   support

      ATTACK       0.97      0.81      0.88      8021
      NORMAL       0.40      0.82      0.54      1270

    accuracy                           0.81      9291
   macro avg       0.68      0.81      0.71      9291
weighted avg       0.89      0.81      0.83      9291



In [35]:
path_training = "../datasets/training/final_training_dataset_no_history_SMOTENC.csv"
path_testing = "../datasets/testing/final_testing_dataset_no_history.csv"
train_second_classifier_model_grid_search(path_train=path_training,path_test=path_testing,save_model=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

**************************************** the second model Gridsearch results ****************************************
******************** cross validation results  ********************


,iterations,test-MultiClass-mean,test-MultiClass-std,train-MultiClass-mean,train-MultiClass-std
0,0,1.038001,0.006335,1.039910,0.000675
1,1,0.711870,0.020640,0.706970,0.005739
2,2,0.629093,0.009661,0.622526,0.009344
3,3,0.565023,0.012696,0.561611,0.023266
4,4,0.536389,0.015372,0.529649,0.010193
...,...,...,...,...,...
995,995,0.491289,0.025593,0.215288,0.018874
996,996,0.491382,0.025640,0.215228,0.018832
997,997,0.491429,0.025584,0.215212,0.018842
998,998,0.491497,0.025746,0.215194,0.018855


******************** best parameters ********************


{'depth': 8, 'l2_leaf_reg': 7, 'learning_rate': 0.7}

**************************************** the second model report ****************************************


col_0,Broken_Authentication,SENSITIVE_DATA_EXPOSURE,SQL_INJECTION,WEB_SCAN,XSS,brute_force
output_2,,,,,,
Broken_Authentication,56,0,1,0,2,1
SENSITIVE_DATA_EXPOSURE,52,3242,92,1633,173,120
SQL_INJECTION,14,17,1192,12,73,7
WEB_SCAN,0,338,33,697,44,19
XSS,4,1,6,0,52,0
brute_force,8,0,0,0,3,129


************************************************************
                         precision    recall  f1-score   support

  Broken_Authentication       0.42      0.93      0.58        60
SENSITIVE_DATA_EXPOSURE       0.90      0.61      0.73      5312
          SQL_INJECTION       0.90      0.91      0.90      1315
               WEB_SCAN       0.30      0.62      0.40      1131
                    XSS       0.15      0.83      0.25        63
            brute_force       0.47      0.92      0.62       140

               accuracy                           0.67      8021
              macro avg       0.52      0.80      0.58      8021
           weighted avg       0.80      0.67      0.70      8021



In [14]:
path_training = "../datasets/training/final_training_dataset_no_history_SMOTENC.csv"
path_testing = "../datasets/testing/final_testing_dataset_no_history.csv"
dataset = pd.read_csv(path_testing)
print(dataset["_source.agent.description"].value_counts())
print("shape",dataset.shape)

APACHE_SERVER    9291
Name: _source.agent.description, dtype: int64
shape (9291, 18)


In [45]:
path_training = "../datasets/training/final_training_dataset_no_history_SMOTENC.csv"
path_testing = "../datasets/testing/final_testing_dataset_no_history.csv"
train_multi_classifier_model(path_train=path_training,path_test=path_testing,save_model=True)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

**************************************** the first model report ****************************************


col_0,ATTACK,NORMAL
output_1,,
ATTACK,6522,1499
NORMAL,247,1023


************************************************************
              precision    recall  f1-score   support

      ATTACK       0.96      0.81      0.88      8021
      NORMAL       0.41      0.81      0.54      1270

    accuracy                           0.81      9291
   macro avg       0.68      0.81      0.71      9291
weighted avg       0.89      0.81      0.84      9291

**************************************** the second model report ****************************************


col_0,Broken_Authentication,SENSITIVE_DATA_EXPOSURE,SQL_INJECTION,WEB_SCAN,XSS,brute_force
output_2,,,,,,
Broken_Authentication,57,0,0,0,2,1
SENSITIVE_DATA_EXPOSURE,51,3246,96,1627,203,89
SQL_INJECTION,14,7,1197,2,88,7
WEB_SCAN,0,333,37,691,51,19
XSS,4,0,6,0,53,0
brute_force,8,1,0,2,3,126


************************************************************
                         precision    recall  f1-score   support

  Broken_Authentication       0.43      0.95      0.59        60
SENSITIVE_DATA_EXPOSURE       0.90      0.61      0.73      5312
          SQL_INJECTION       0.90      0.91      0.90      1315
               WEB_SCAN       0.30      0.61      0.40      1131
                    XSS       0.13      0.84      0.23        63
            brute_force       0.52      0.90      0.66       140

               accuracy                           0.67      8021
              macro avg       0.53      0.80      0.58      8021
           weighted avg       0.80      0.67      0.71      8021



In [78]:
model_1.save_model("catboost_model_1.bin")
model_1.save_model("catboost_model_1.json",format='json')

In [79]:
model_2.save_model("catboost_model_2.bin")
model_2.save_model("catboost_model_2.json",format='json')


In [ ]:
CatBoostClassifier()